# 04 — iMAGE versioned fine-tuning and evaluation

This notebook creates an isolated experiment folder for each run. It supports Windows CUDA, Apple Silicon MPS, and CPU. It never overwrites `weights/` or automatically deploys checkpoints. Run cells from top to bottom.


## 1. Project setup and device check


In [9]:
from pathlib import Path
from datetime import datetime
import json, sys, torch
ROOT = Path.cwd().resolve()
if not (ROOT / 'app').is_dir(): ROOT = ROOT.parent
if not (ROOT / 'app').is_dir(): raise FileNotFoundError('Open the CCS250_CompVis_Final project root in VS Code/Jupyter.')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
from app.experiments import setup_device, dataset_counts, fine_tune, evaluate_model
from app.models import NAMES
from app.online import pretrained_path
DEVICE = setup_device()
print('Project:', ROOT)
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', dataset_counts())


Project: C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final
PyTorch: 2.11.0+cu128 | Device: cuda
GPU: NVIDIA GeForce RTX 4070
Dataset: {'train': 4000, 'val': 500, 'test': 500}


## 2. Create a unique experiment folder

The folder name is generated once per notebook run. Keep this value for every training and evaluation cell in this experiment.


In [16]:
EXPERIMENT_ID = 'finetune_' + datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_DIR = ROOT / 'experiments' / EXPERIMENT_ID
(EXPERIMENT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=False)
(EXPERIMENT_DIR / 'logs').mkdir(parents=True, exist_ok=True)
(EXPERIMENT_DIR / 'evaluation').mkdir(parents=True, exist_ok=True)
print('Experiment:', EXPERIMENT_ID)
print('Output:', EXPERIMENT_DIR)


Experiment: finetune_20261116_155735
Output: C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\experiments\finetune_20261116_155735


## 3. Choose initialization and per-model training settings

`pretrained` starts from upstream published checkpoints. `deployed` continues from `weights/<model>.pth`. `experiment` continues from a previous experiment folder. Continuing from a weights-only checkpoint restarts the optimizer; it does not restore AdamW momentum or scheduler state.

**Per-model parameters:** every model starts from `DEFAULTS`. To change a setting for one model only, add it to that model's entry in `MODEL_OVERRIDES`. Anything you leave out falls back to the default. `RUN_MODE='smoke'` forces tiny epoch/step/validation limits on all models; `'full'` uses the values below.


In [17]:
INITIALIZATION = 'deployed'  # 'pretrained', 'deployed', or 'experiment'
SOURCE_EXPERIMENT = 'finetune_YYYYMMDD_HHMMSS'  # used only for INITIALIZATION='experiment'
RUN_MODE = 'full'  # first use 'smoke'; after it passes, change to 'full'
MODELS_TO_TRAIN = ['dncnn', 'nafnet', 'restormer']

# ---- Settings shared by every model unless overridden below ----
DEFAULTS = {
    'patch': 128,
    'batch_size': 2 if DEVICE.type == 'cuda' else 1,
    'epochs': 5,
    'learning_rate': 1e-5,
    'patience': 3,
    'seed': 42,
}

# ---- Per-model overrides: list ONLY what differs from DEFAULTS ----
# Example (uncomment and edit):
#   'dncnn':     {'learning_rate': 1e-4, 'batch_size': 16, 'epochs': 30, 'patience': 5},
#   'nafnet':    {'learning_rate': 1e-4, 'batch_size': 4,  'epochs': 30, 'patience': 5},
#   'restormer': {'learning_rate': 5e-5, 'batch_size': 2,  'epochs': 30, 'patience': 5},
MODEL_OVERRIDES = {
    'dncnn':     {'learning_rate': 1e-4, 'batch_size': 16, 'epochs': 30, 'patience': 5},
   'nafnet':    {'learning_rate': 1e-4, 'batch_size': 4,  'epochs': 30, 'patience': 5},
   'restormer': {'learning_rate': 5e-5, 'batch_size': 2,  'epochs': 30, 'patience': 5},
}

# ---- Run-mode limits (applied to all models) ----
RUN_LIMITS = {
    'smoke': {'epochs': 1, 'max_train_steps': 5, 'validation_limit': 10},
    'full':  {'max_train_steps': 0, 'validation_limit': 500},
}

assert INITIALIZATION in ('pretrained', 'deployed', 'experiment')
assert RUN_MODE in ('smoke', 'full')
assert set(MODELS_TO_TRAIN).issubset(NAMES)
unknown_models = set(MODEL_OVERRIDES) - set(NAMES)
if unknown_models: raise ValueError(f'MODEL_OVERRIDES has unknown models: {sorted(unknown_models)}')

MODEL_PARAMS = {}
for model_name in MODELS_TO_TRAIN:
    overrides = MODEL_OVERRIDES.get(model_name, {})
    bad_keys = set(overrides) - set(DEFAULTS)
    if bad_keys: raise ValueError(f'{model_name}: unknown parameter(s) {sorted(bad_keys)}. Allowed: {sorted(DEFAULTS)}')
    params = {**DEFAULTS, **overrides}
    params.update(RUN_LIMITS[RUN_MODE])
    MODEL_PARAMS[model_name] = params

SEED = DEFAULTS['seed']  # used by the evaluation cell

def source_checkpoint(name):
    if INITIALIZATION == 'pretrained': return pretrained_path(name)
    if INITIALIZATION == 'deployed':
        path = ROOT / 'weights' / f'{name}.pth'
    else:
        path = ROOT / 'experiments' / SOURCE_EXPERIMENT / 'checkpoints' / f'{name}.pth'
    if not path.is_file(): raise FileNotFoundError(f'Missing selected starting checkpoint: {path}')
    return path

print('Initialization:', INITIALIZATION, '| Run:', RUN_MODE, '| Models:', MODELS_TO_TRAIN)
for model_name in MODELS_TO_TRAIN:
    print(f'\n[{model_name}] starts from', source_checkpoint(model_name))
    for key, value in MODEL_PARAMS[model_name].items():
        marker = '*' if key in MODEL_OVERRIDES.get(model_name, {}) else ' '
        print(f'  {marker} {key}: {value}')
print('\n(* = overridden for this model)')


Initialization: deployed | Run: full | Models: ['dncnn', 'nafnet', 'restormer']

[dncnn] starts from C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\weights\dncnn.pth
    patch: 128
  * batch_size: 16
  * epochs: 30
  * learning_rate: 0.0001
  * patience: 5
    seed: 42
    max_train_steps: 0
    validation_limit: 500

[nafnet] starts from C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\weights\nafnet.pth
    patch: 128
  * batch_size: 4
  * epochs: 30
  * learning_rate: 0.0001
  * patience: 5
    seed: 42
    max_train_steps: 0
    validation_limit: 500

[restormer] starts from C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\weights\restormer.pth
    patch: 128
  * batch_size: 2
  * epochs: 30
  * learning_rate: 5e-05
  * patience: 5
    seed: 42
    max_train_steps: 0
    validation_limit: 500

(* = overridden for this model)


## 4. Fine-tune models into this experiment

Run this cell after reviewing the settings. Each model is trained with its own entry from `MODEL_PARAMS` and saves its best validation checkpoint to `EXPERIMENT_DIR/checkpoints/` and its epoch log to `EXPERIMENT_DIR/logs/`. The full per-model settings are recorded in `config.json`. Re-running this cell for the same experiment overwrites that experiment's checkpoint files, not production weights. Create a new experiment ID for a separate run.


In [18]:
training_config = {
    'experiment_id': EXPERIMENT_ID, 'initialization': INITIALIZATION,
    'source_experiment': SOURCE_EXPERIMENT if INITIALIZATION == 'experiment' else None,
    'device': str(DEVICE), 'pytorch': torch.__version__, 'models': MODELS_TO_TRAIN,
    'run_mode': RUN_MODE, 'defaults': DEFAULTS, 'model_overrides': MODEL_OVERRIDES,
    'model_params': MODEL_PARAMS, 'manifest': 'data/splits.csv'
}
(EXPERIMENT_DIR / 'config.json').write_text(json.dumps(training_config, indent=2), encoding='utf8')
histories = {}
for model_name in MODELS_TO_TRAIN:
    p = MODEL_PARAMS[model_name]
    print(f'=== Training {model_name} ===', p)
    histories[model_name] = fine_tune(
        model_name, output_dir=EXPERIMENT_DIR, checkpoint=source_checkpoint(model_name),
        epochs=p['epochs'], batch_size=p['batch_size'], patch=p['patch'], lr=p['learning_rate'],
        validation_limit=p['validation_limit'], max_train_steps=p['max_train_steps'],
        seed=p['seed'], patience=p['patience']
    )
print('Training complete:', EXPERIMENT_DIR)


=== Training dncnn === {'patch': 128, 'batch_size': 16, 'epochs': 30, 'learning_rate': 0.0001, 'patience': 5, 'seed': 42, 'max_train_steps': 0, 'validation_limit': 500}
dncnn: cuda; initialized from C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\weights\dncnn.pth; {'train': 4000, 'val': 500, 'test': 500}


dncnn 1/30: 100%|██████████| 250/250 [00:50<00:00,  4.94it/s]


{'epoch': 1, 'train_l1': 0.026559085559099912, 'validation_psnr_db': 36.507431816101075, 'elapsed_min': 0.937}


dncnn 2/30: 100%|██████████| 250/250 [00:49<00:00,  5.01it/s]


{'epoch': 2, 'train_l1': 0.014633979197591544, 'validation_psnr_db': 35.867173126220706, 'elapsed_min': 1.861}


dncnn 3/30: 100%|██████████| 250/250 [00:49<00:00,  5.05it/s]


... (53 outputs left out)

restormer 8/30: 100%|██████████| 2000/2000 [05:04<00:00,  6.56it/s]


{'epoch': 8, 'train_l1': 0.0074684275821782645, 'validation_psnr_db': 42.33304773712158, 'elapsed_min': 42.853}


restormer 9/30: 100%|██████████| 2000/2000 [05:04<00:00,  6.57it/s]


{'epoch': 9, 'train_l1': 0.007345459339558147, 'validation_psnr_db': 42.373021232604984, 'elapsed_min': 48.218}
Early stopping
Best validation PSNR: 42.5395 dB; checkpoint: C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\experiments\finetune_20261116_155735\checkpoints\restormer.pth
Training complete: C:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\experiments\finetune_20261116_155735


## 5. Evaluate on the untouched test split

For final reporting, use `RUN_MODE='full'` and `LIMIT=0` to evaluate all 500 test images for every model. A smoke test is only a pipeline check and must not be used for model selection or deployment.


In [19]:
RUN_EVALUATION = True  # set True after full training has completed
LIMIT = 0  # 0 = all 500 test images; nonzero is smoke testing only
TEST_PATCH = 256
TILE = 256
summaries = []
if RUN_EVALUATION:
    if RUN_MODE != 'full' or LIMIT != 0: raise ValueError('Final evaluation requires full training and LIMIT=0.')
    for model_name in MODELS_TO_TRAIN:
        checkpoint = EXPERIMENT_DIR / 'checkpoints' / f'{model_name}.pth'
        if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
        summaries.append(evaluate_model(model_name, checkpoint=checkpoint, output_dir=EXPERIMENT_DIR, patch=TEST_PATCH, seed=SEED, limit=LIMIT, tile=TILE))
else:
    print('Evaluation skipped. Set RUN_EVALUATION=True after full training.')


Evaluating dncnn: 100%|██████████| 500/500 [00:13<00:00, 37.91it/s]


{
  "model": "dncnn",
  "checkpoint": "C:\\Users\\rmclo\\OneDrive\\Documents\\GitHub\\CCS250_CompVis_Final\\experiments\\finetune_20261116_155735\\checkpoints\\dncnn.pth",
  "checkpoint_sha256": "02d1ae8e04c48685b7ed44ed2b52152252c30795f4b55b85f44185f702693c03",
  "manifest_sha256": "574b2f63fd3c509c8f4b11ab9a82520195bb27bc0235abc9a67ea52b8ff67ea6",
  "split": "test",
  "number_of_test_images": 500,
  "limited_test": false,
  "patch": 256,
  "seed": 42,
  "tile": 256,
  "device": "cuda",
  "psnr_db": 38.0881,
  "ssim": 0.92051,
  "mean_runtime_ms": 15.23,
  "parameters_m": 0.668
}


Evaluating nafnet: 100%|██████████| 500/500 [00:18<00:00, 27.70it/s]


{
  "model": "nafnet",
  "checkpoint": "C:\\Users\\rmclo\\OneDrive\\Documents\\GitHub\\CCS250_CompVis_Final\\experiments\\finetune_20261116_155735\\checkpoints\\nafnet.pth",
  "checkpoint_sha256": "e1859d072467ba30995d05a9a0d269b4abbea891b8aa4cf5427de815f3856494",
  "manifest_sha256": "574b2f63fd3c509c8f4b11ab9a82520195bb27bc0235abc9a67ea52b8ff67ea6",
  "split": "test",
  "number_of_test_images": 500,
  "limited_test": false,
  "patch": 256,
  "seed": 42,
  "tile": 256,
  "device": "cuda",
  "psnr_db": 41.3955,
  "ssim": 0.95532,
  "mean_runtime_ms": 26.25,
  "parameters_m": 29.16
}


Evaluating restormer: 100%|██████████| 500/500 [00:52<00:00,  9.54it/s]


{
  "model": "restormer",
  "checkpoint": "C:\\Users\\rmclo\\OneDrive\\Documents\\GitHub\\CCS250_CompVis_Final\\experiments\\finetune_20261116_155735\\checkpoints\\restormer.pth",
  "checkpoint_sha256": "c2c1404224e7b9124bef7b83d2dc9f6a0d79d926e73d8f781295729c22094aed",
  "manifest_sha256": "574b2f63fd3c509c8f4b11ab9a82520195bb27bc0235abc9a67ea52b8ff67ea6",
  "split": "test",
  "number_of_test_images": 500,
  "limited_test": false,
  "patch": 256,
  "seed": 42,
  "tile": 256,
  "device": "cuda",
  "psnr_db": 40.1914,
  "ssim": 0.94806,
  "mean_runtime_ms": 94.63,
  "parameters_m": 26.112
}


## 6. Compare and mark the experiment as evaluated

An experiment is eligible for selection only when all three models have full 500-image test summaries. This cell writes a small manifest; it does not change the app's active checkpoint directory.


In [20]:
if RUN_EVALUATION:
    import pandas as pd
    frame = pd.DataFrame(summaries)
    display(frame[['model', 'number_of_test_images', 'psnr_db', 'ssim', 'mean_runtime_ms', 'device']])
    if len(summaries) != 3 or any(s['number_of_test_images'] != 500 or s['limited_test'] for s in summaries):
        raise ValueError('All three models must have complete 500-image evaluations.')
    hashes = {s['manifest_sha256'] for s in summaries}
    if len(hashes) != 1: raise ValueError('Models were not evaluated against the same manifest.')
    selection = {'experiment_id': EXPERIMENT_ID, 'status': 'evaluated', 'models': {s['model']: {'checkpoint': f'checkpoints/{s["model"]}.pth', 'checkpoint_sha256': s['checkpoint_sha256'], 'psnr_db': s['psnr_db'], 'ssim': s['ssim'], 'mean_runtime_ms': s['mean_runtime_ms']} for s in summaries}}
    (EXPERIMENT_DIR / 'experiment.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
    print('Experiment evaluated. Review visual outputs before selecting it in the launcher.')
else:
    print('No evaluation manifest created yet.')


,model,number_of_test_images,psnr_db,ssim,mean_runtime_ms,device
0,dncnn,500,38.0881,0.92051,15.23,cuda
1,nafnet,500,41.3955,0.95532,26.25,cuda
2,restormer,500,40.1914,0.94806,94.63,cuda


Experiment evaluated. Review visual outputs before selecting it in the launcher.


## 7. Run the app with this experiment (manual selection)

Only select an experiment after full evaluation and visual inspection. The app expects all three files named `dncnn.pth`, `nafnet.pth`, and `restormer.pth` in the chosen folder. This notebook never copies files into `weights/`.

macOS: `./scripts/run_app.sh experiments/EXPERIMENT_ID/checkpoints`

Windows PowerShell: `./scripts/run_app.ps1 -CheckpointDir experiments/EXPERIMENT_ID/checkpoints`
